# 02 · Zaawansowany chunking: tabela, diagram i obraz

Zwykły podział po znakach może oddzielić wiersz tabeli od nagłówka kolumn albo zgubić obraz. Tu zachowujemy **typ fragmentu, strukturę i pochodzenie**. Dokument Acme ma prawdziwą tabelę Markdown; diagram tworzymy lokalnie jako jawnie oznaczoną ilustrację metody, a model wizyjny opisuje to, co widzi.

Obrazowy opis modelu jest hipotezą do weryfikacji. Nie zastępuje oryginalnego pliku ani nie zamienia ilustracji w źródło danych finansowych.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import base64
import os
import sys
from pathlib import Path

import httpx
import numpy as np
from IPython.display import Image, display
from dotenv import dotenv_values
from ollama import Client
from PIL import Image as PILImage, ImageDraw, ImageFont

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import load_document, ollama_settings

## 1. Wiersz tabeli musi znać swoje kolumny

W raporcie nagłówek tabeli wyjaśnia, czy `892,184.17` oznacza liczbę faktur, kwotę czy datę. Jeden wiersz jako samodzielny tekst jest nieczytelny. Kopiujemy nagłówki do każdego rekordu i zachowujemy ID dokumentu, wersję oraz sekcję. Separator `---` w Markdown nie jest danymi.

In [ ]:
doc = load_document("FIN-YTD-2011")
table = doc["body"].split("## Monthly evidence table", 1)[1].split("## ", 1)[0]
lines = [line for line in table.splitlines() if line.startswith("|")]
columns = [cell.strip() for cell in lines[0].strip("|").split("|")]
rows = [[cell.strip() for cell in line.strip("|").split("|")] for line in lines[2:]]
table_chunks = [
    {"id": f"FIN-YTD-2011:v1:table:r{i:02d}", "modality": "table_row", "source": doc["path"].relative_to(ROOT).as_posix(),
     "text": "Monthly evidence table. " + "; ".join(f"{key}: {value}" for key, value in zip(columns, row))}
    for i, row in enumerate(rows)
]
table_chunks[2]

Odpowiedź o listopadowym billing może teraz trafić w jeden wiersz, ale wyjaśnienie różnicy z październikiem wymaga **dwóch** wierszy i akapitu interpretacji. Tabelę często indeksuje się dwutorowo: rekordy wierszy do dokładnego trafienia oraz opis całej tabeli do szerszego pytania.

In [ ]:
table_summary = {"id": "FIN-YTD-2011:v1:table:summary", "modality": "table_summary",
                 "source": doc["path"].relative_to(ROOT).as_posix(),
                 "text": "Monthly evidence table: invoice count and header billing GBP for September, October, November and partial December 2011. December is incomplete."}
[table_summary, table_chunks[1], table_chunks[2]]

## 2. Diagram: piksele + podpis + relacja do źródła

Tworzymy obraz z etapami backtestu. To **ilustracja sporządzona na potrzeby lekcji** na podstawie notatki `SCI-FORECAST-2011:v1`, nie nowy wynik badania. W systemie RAG zapisujemy ścieżkę obrazu, podpis, pochodzenie i wersję. Gdy pytanie wymaga liczb lub metodologii, wracamy do tekstu źródłowego.

In [ ]:
image_path = ROOT / ".tmp" / "rag" / "forecast_flow.png"
image_path.parent.mkdir(parents=True, exist_ok=True)
canvas = PILImage.new("RGB", (960, 230), "white")
draw = ImageDraw.Draw(canvas)
font = ImageFont.load_default(size=16)
labels = ["Freeze cutoff", "Train baseline", "Forecast next period", "Compare actuals"]
for i, label in enumerate(labels):
    left = 18 + i * 238
    draw.rounded_rectangle((left, 65, left + 205, 155), radius=14, outline="#2456A6", width=3, fill="#EAF2FF")
    draw.text((left + 12, 101), label, fill="#15305A", font=font)
    if i < 3:
        draw.line((left + 207, 110, left + 233, 110), fill="#2456A6", width=4)
        draw.polygon([(left + 233, 110), (left + 222, 103), (left + 222, 117)], fill="#2456A6")
canvas.save(image_path)
display(Image(filename=str(image_path)))

## 3. Model wizyjny odczytuje obraz

`gemma4:12b` ma lokalną funkcję vision i chat; nazwę można zmienić przez `RAG_VISION_MODEL` w **root `.env`**. `clef` i `clef-flash` są modelami **decyzyjnymi System One**, więc nie używamy ich endpointu chat do podpisów. Wynik sprawdzamy z etykietami, które sami narysowaliśmy. Gdy model pominie lub przestawi etap, popraw podpis przed indeksowaniem. W prawdziwym PDF analogicznie zachowaj numer strony, współrzędne i sąsiedni podpis.

In [ ]:
host, _ = ollama_settings()
vision_model = os.environ.get("RAG_VISION_MODEL") or dotenv_values(ROOT / ".env").get("RAG_VISION_MODEL") or "gemma4:12b"
vision_caption = Client(host=host).chat(
    model=vision_model,
    messages=[{"role": "user", "content": "Transcribe the four box labels from left to right, then describe the arrows in one sentence. Do not infer financial results.",
               "images": [str(image_path)]}],
    options={"temperature": 0}, think=False,
).message.content.strip()
missing_labels = [label for label in labels if label.lower() not in vision_caption.lower()]
{"model_caption": vision_caption, "missing_labels": missing_labels}

`missing_labels` pokazuje, czy model odczytał wszystkie etapy. Diagram stworzyliśmy sami, więc opis do indeksu budujemy z **jawnych etykiet i strzałek z kodu**, a surowy podpis modelu zachowujemy do porównania. Przy obcym obrazie nie mamy takiego źródła etykiet: potrzebna jest kontrola człowieka lub OCR zweryfikowany z obrazem.

In [ ]:
# Diagram powstał z jawnych etykiet i strzałek: ten opis można zweryfikować bez modelu.
verified_caption = "Four boxes from left to right: " + " -> ".join(labels) + ". Arrows connect adjacent boxes."
diagram_chunk = {"id": "SCI-FORECAST-2011:v1:diagram:lesson", "modality": "diagram_caption",
                 "source": "SCI-FORECAST-2011:v1 (illustration, not source evidence)",
                 "image_path": image_path.relative_to(ROOT).as_posix(), "text": verified_caption,
                 "model_caption": vision_caption, "model_omitted_labels": missing_labels}
{"types": [chunk["modality"] for chunk in [table_summary, table_chunks[0], diagram_chunk]],
 "indexed_diagram_caption": diagram_chunk["text"], "model_omitted_labels": missing_labels}

## 4. Prawdziwe embeddingi obrazu i tekstu w jednej przestrzeni

`embeddinggemma-2:latest` przyjmuje obraz i tekst przez `/api/embed` i zwraca zgodne wektory 768D. Biblioteka Python Ollama w tej wersji typuje `Client.embed` jako tekstowe wejście, więc do multimodalnego żądania używamy jawnie oficjalnego HTTP API. Porównujemy wektor **pikseli diagramu** z dwoma tekstami. To inny model i inna przestrzeń niż `qwen3-embedding:0.6b`; nigdy nie mieszaj ich wektorów w jednym indeksie.

In [ ]:
multimodal_model = os.environ.get("RAG_MULTIMODAL_EMBED_MODEL") or dotenv_values(ROOT / ".env").get("RAG_MULTIMODAL_EMBED_MODEL") or "embeddinggemma-2:latest"
image_base64 = base64.b64encode(image_path.read_bytes()).decode("ascii")
payload = {"model": multimodal_model, "input": [
    {"image": image_base64},
    {"text": "A four-step diagram: freeze cutoff, train baseline, forecast next period, compare actuals"},
    {"text": "A ledger of customer payment receipts and invoice allocations"},
], "truncate": False}
response = httpx.post(host.rstrip("/") + "/api/embed", json=payload, timeout=180)
response.raise_for_status()
mixed_vectors = np.asarray(response.json()["embeddings"], dtype=np.float32)
mixed_unit = mixed_vectors / np.linalg.norm(mixed_vectors, axis=1, keepdims=True)
{"shape": mixed_vectors.shape, "diagram_vs_matching_text": float(mixed_unit[0] @ mixed_unit[1]),
 "diagram_vs_unrelated_text": float(mixed_unit[0] @ mixed_unit[2])}

## 5. Granica multimodalna

Mamy dwie drogi: (1) obraz → sprawdzony podpis → tekstowy embedding; (2) piksele i tekst → **wspólna przestrzeń** modelu multimodalnego. Surowy podpis modelu może zgubić układ, strzałki lub drobny tekst. Wektor obrazu też nie jest dowodem, że model odczytał każdą etykietę. Przechowuj oryginał i pokaż go użytkownikowi.

**Ćwiczenie:** narysuj połączenie zwrotne między etapami, ponów podpis i sprawdź, czy model je zauważa. Dla tabeli usuń nagłówek z tekstu wiersza i porównaj interpretację liczby `892,184.17`.